# LSTM — PyTorch ECG Pipeline (Part A)

## Model: Long Short-Term Memory (LSTM) vs GRU
- **Dataset**: ECG5000 (augmented) — 7,250 train / 1,000 test, 140 timesteps, 5 heartbeat classes
- **Task**: Classify heartbeat arrhythmias — break RNN's 0.55 macro F1 ceiling via augmentation + LSTM
- **Framework showcase**: LSTM vs GRU head-to-head on augmented data, cell state analysis, augmentation impact

## Evaluation Strategy
- **Primary metric**: Macro F1 (not accuracy — class imbalance still present despite augmentation)
- **Key comparison**: RNN #12 GRU-128 on original data (0.55 F1) vs LSTM on augmented data
- **Gradient analysis**: LSTM cell state gradient pathway vs GRU

## Pipeline
1. Load augmented data + config
2. GRU-128 baseline on augmented data (control)
3. LSTM-128 model
4. Architecture sweep (LSTM-64, LSTM-128, LSTM-64x3, BiLSTM-64)
5. LSTM vs GRU head-to-head comparison
6. Augmentation impact analysis
7. Gradient flow + hidden/cell state analysis
8. Training visualization
9. Performance benchmarks
10. Save results
11. MLflow + model export

In [1]:
# Step 1: Setup
"""
Load augmented ECG5000, configure GPU, prepare DataLoaders.
Augmented training set: 7,250 samples (was 4,000).
Minority classes boosted to ~1,167 each via jitter/scaling/time_warp.
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import sys
import os
sys.path.append('../..')

from utils.data_loader import load_processed_data
from utils.metrics import evaluate_classifier, macro_f1_score
from utils.rnn_utils import compute_gradient_norms, extract_hidden_states
from utils.visualization import (plot_training_history,
                                  plot_confusion_matrix_multiclass,
                                  plot_gradient_flow,
                                  plot_ecg_predictions,
                                  plot_hidden_state_evolution)
from utils.performance import track_performance, track_inference, get_model_size
from utils.results import build_results_dict, save_results, add_result, print_comparison

# Config
RANDOM_STATE = 113
FRAMEWORK = "PyTorch"
MODEL_NAME = "LSTM"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE = 64
N_CLASSES = 5
SEQ_LEN = 140
N_FEATURES = 1

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(RANDOM_STATE)

# Load augmented ECG data
X_train, X_test, y_train, y_test, metadata = load_processed_data('lstm/ecg')

CLASS_NAMES = metadata['class_names']

# Class weights from augmented distribution (keys are strings in JSON)
class_weights = {int(k): v for k, v in metadata['class_weights_augmented'].items()}
class_weights_tensor = torch.tensor(
    [class_weights[i] for i in range(N_CLASSES)],
    dtype=torch.float32
).to(device)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

# DataLoader
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# Original counts for comparison (from RNN #12)
orig_counts = [2335, 1414, 77, 155, 19]
aug_counts = [int(np.sum(y_train == i)) for i in range(N_CLASSES)]

print("=" * 60)
print(f"[1/11] {FRAMEWORK} — {MODEL_NAME} ECG Pipeline")
print("=" * 60)
print(f"Device: {device} ({torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'})")
print(f"Train: {X_train_t.shape} (augmented from 4,000)")
print(f"Test: {X_test_t.shape} (unchanged)")
print(f"Sequence: {SEQ_LEN} timesteps x {N_FEATURES} feature")
print(f"Classes: {N_CLASSES} ({', '.join(CLASS_NAMES)})")
print(f"\nAugmentation impact:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<15} {orig_counts[i]:>5} -> {aug_counts[i]:>5} (+{aug_counts[i] - orig_counts[i]})")
print(f"\nRNN #12 best: GRU-128, 91.8% acc, 0.55 macro F1 (on original 4K)")
print(f"Goal: break 0.55 macro F1 ceiling with augmentation + LSTM")

[1/11] PyTorch — LSTM ECG Pipeline
Device: cuda (NVIDIA GeForce RTX 4090)
Train: torch.Size([7250, 140, 1]) (augmented from 4,000)
Test: torch.Size([1000, 140, 1]) (unchanged)
Sequence: 140 timesteps x 1 feature
Classes: 5 (Normal, R-on-T PVC, PVC, SP, UB)

Augmentation impact:
  Normal           2335 ->  2335 (+0)
  R-on-T PVC       1414 ->  1414 (+0)
  PVC                77 ->  1167 (+1090)
  SP                155 ->  1167 (+1012)
  UB                 19 ->  1167 (+1148)

RNN #12 best: GRU-128, 91.8% acc, 0.55 macro F1 (on original 4K)
Goal: break 0.55 macro F1 ceiling with augmentation + LSTM


In [2]:
# Step 2: GRU Baseline on Augmented Data
"""
Control experiment: same GRU-128 architecture from RNN #12,
but trained on augmented data (7,250 samples instead of 4,000).
Isolates the effect of augmentation from architecture change.
If GRU improves on augmented data, augmentation is the driver.
If LSTM improves further, architecture matters too.
"""

print("=" * 60)
print("[2/11] GRU-128 Baseline on Augmented Data")
print("=" * 60)


class GRUClassifier(nn.Module):
    # GRU for sequence classification (same as RNN)
    def __init__(self, input_size, hidden_size, num_layers, n_classes):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                           batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        output, h_n = self.gru(x)
        return self.fc(output[:, -1, :])


def train_sequence_model(model, criterion, max_epochs=50, patience=10, lr=1e-3):
    # Train RNN/GRU/LSTM with early stopping on val macro F1
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Train/val split (10%)
    n_val = int(len(X_train_t) * 0.1)
    perm = torch.randperm(len(X_train_t), device=device,
                           generator=torch.Generator(device=device).manual_seed(RANDOM_STATE))
    val_idx, tr_idx = perm[:n_val], perm[n_val:]

    X_val = X_train_t[val_idx]
    y_val = y_train_t[val_idx]

    tr_ds = TensorDataset(X_train_t[tr_idx], y_train_t[tr_idx])
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)

    best_val_f1 = 0.0
    wait = 0
    best_state = None
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        for batch_x, batch_y in tr_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_y)
            correct += (logits.argmax(1) == batch_y).sum().item()
            total += len(batch_y)

        train_losses.append(epoch_loss / total)
        train_accs.append(correct / total)

        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss = criterion(val_logits, y_val).item()
            val_preds = val_logits.argmax(1).cpu().numpy()
            val_acc = float((val_preds == y_val.cpu().numpy()).mean())
            val_f1 = float(macro_f1_score(y_val.cpu().numpy(), val_preds))

        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            wait = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)

    return {
        'train_loss': train_losses, 'val_loss': val_losses,
        'train_acc': train_accs, 'val_acc': val_accs,
        'epochs': len(train_losses), 'best_val_f1': best_val_f1
    }


# Build and train GRU-128 on augmented data
gru_model = GRUClassifier(N_FEATURES, hidden_size=128, num_layers=2,
                            n_classes=N_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
n_params_gru = sum(p.numel() for p in gru_model.parameters())

print(f"Architecture: GRU(1, 128, 2 layers) -> FC(128, 5)")
print(f"Parameters: {n_params_gru:,}")

with track_performance(gpu=True) as perf_gru:
    hist_gru = train_sequence_model(gru_model, criterion, max_epochs=50, patience=10)
    torch.cuda.synchronize()

# Test evaluation
gru_model.eval()
with torch.no_grad():
    gru_preds = gru_model(X_test_t).argmax(1).cpu().numpy()

gru_metrics = evaluate_classifier(y_test, gru_preds)
gru_f1, gru_per_class = macro_f1_score(y_test, gru_preds, return_per_class=True)

print(f"\nEpochs: {hist_gru['epochs']} | Best val F1: {hist_gru['best_val_f1']:.4f}")
print(f"Training time: {perf_gru['time']:.2f}s")
print(f"\nTest Results:")
print(f"  Accuracy:  {gru_metrics['accuracy']:.4f}")
print(f"  Macro F1:  {gru_f1:.4f}")
print(f"\nPer-class F1:")
for i, (name, f1) in enumerate(zip(CLASS_NAMES, gru_per_class)):
    rnn_f1 = [0.9732, 0.9399, 0.4242, 0.3333, 0.0690][i]  # RNN #12 GRU-128 results
    delta = f1 - rnn_f1
    print(f"  {name:<15} F1={f1:.4f} (RNN #12: {rnn_f1:.4f}, delta: {delta:+.4f})")

print(f"\nRNN #12 GRU-128 (original 4K): macro F1 = 0.5479")
print(f"This GRU-128 (augmented 7.25K): macro F1 = {gru_f1:.4f} ({gru_f1 - 0.5479:+.4f})")

[2/11] GRU-128 Baseline on Augmented Data
Architecture: GRU(1, 128, 2 layers) -> FC(128, 5)
Parameters: 150,021

Epochs: 50 | Best val F1: 0.8726
Training time: 17.46s

Test Results:
  Accuracy:  0.9150
  Macro F1:  0.5950

Per-class F1:
  Normal          F1=0.9845 (RNN #12: 0.9732, delta: +0.0113)
  R-on-T PVC      F1=0.9184 (RNN #12: 0.9399, delta: -0.0215)
  PVC             F1=0.5000 (RNN #12: 0.4242, delta: +0.0758)
  SP              F1=0.3371 (RNN #12: 0.3333, delta: +0.0038)
  UB              F1=0.2353 (RNN #12: 0.0690, delta: +0.1663)

RNN #12 GRU-128 (original 4K): macro F1 = 0.5479
This GRU-128 (augmented 7.25K): macro F1 = 0.5950 (+0.0471)


In [3]:
# Step 3: LSTM-128 Model
"""
nn.LSTM adds cell state (c) on top of GRU's hidden state (h).
Same 128-hidden, 2-layer config for fair comparison.
LSTM has ~4x gate matrices per layer vs GRU's 3x, so more params.
"""

print("=" * 60)
print("[3/11] LSTM-128 Model")
print("=" * 60)


class LSTMClassifier(nn.Module):
    """
    LSTM for sequence classification.

    Args:
        input_size: Features per timestep (1 for univariate ECG).
        hidden_size: LSTM hidden dimension.
        num_layers: Stacked LSTM layers.
        n_classes: Output classes.
    """
    def __init__(self, input_size, hidden_size, num_layers, n_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        output, (h_n, c_n) = self.lstm(x)
        return self.fc(output[:, -1, :])


# Build and train LSTM-128
lstm_model = LSTMClassifier(N_FEATURES, hidden_size=128, num_layers=2,
                              n_classes=N_CLASSES).to(device)
criterion_lstm = nn.CrossEntropyLoss(weight=class_weights_tensor)
n_params_lstm = sum(p.numel() for p in lstm_model.parameters())

print(f"Architecture: LSTM(1, 128, 2 layers) -> FC(128, 5)")
print(f"Parameters: {n_params_lstm:,} (vs GRU-128: {n_params_gru:,})")
print(f"  LSTM has 4 gate matrices vs GRU's 3 -> ~33% more params")

with track_performance(gpu=True) as perf_lstm:
    hist_lstm = train_sequence_model(lstm_model, criterion_lstm,
                                      max_epochs=50, patience=10)
    torch.cuda.synchronize()

# Test evaluation
lstm_model.eval()
with torch.no_grad():
    lstm_preds = lstm_model(X_test_t).argmax(1).cpu().numpy()

lstm_metrics = evaluate_classifier(y_test, lstm_preds)
lstm_f1, lstm_per_class = macro_f1_score(y_test, lstm_preds, return_per_class=True)

print(f"\nEpochs: {hist_lstm['epochs']} | Best val F1: {hist_lstm['best_val_f1']:.4f}")
print(f"Training time: {perf_lstm['time']:.2f}s")
print(f"\nTest Results:")
print(f"  Accuracy:  {lstm_metrics['accuracy']:.4f}")
print(f"  Macro F1:  {lstm_f1:.4f}")
print(f"\nPer-class F1 (LSTM vs GRU on augmented data):")
for i, name in enumerate(CLASS_NAMES):
    g_f1 = gru_per_class[i]
    l_f1 = lstm_per_class[i]
    delta = l_f1 - g_f1
    print(f"  {name:<15} GRU={g_f1:.4f}  LSTM={l_f1:.4f}  ({delta:+.4f})")

print(f"\nGRU-128 augmented:  macro F1 = {gru_f1:.4f}")
print(f"LSTM-128 augmented: macro F1 = {lstm_f1:.4f} ({lstm_f1 - gru_f1:+.4f})")

[3/11] LSTM-128 Model
Architecture: LSTM(1, 128, 2 layers) -> FC(128, 5)
Parameters: 199,813 (vs GRU-128: 150,021)
  LSTM has 4 gate matrices vs GRU's 3 -> ~33% more params

Epochs: 50 | Best val F1: 0.8429
Training time: 15.87s

Test Results:
  Accuracy:  0.9130
  Macro F1:  0.5952

Per-class F1 (LSTM vs GRU on augmented data):
  Normal          GRU=0.9845  LSTM=0.9828  (-0.0017)
  R-on-T PVC      GRU=0.9184  LSTM=0.9117  (-0.0066)
  PVC             GRU=0.5000  LSTM=0.5455  (+0.0455)
  SP              GRU=0.3371  LSTM=0.3256  (-0.0115)
  UB              GRU=0.2353  LSTM=0.2105  (-0.0248)

GRU-128 augmented:  macro F1 = 0.5950
LSTM-128 augmented: macro F1 = 0.5952 (+0.0002)
